# Analisi preliminare referti annotati reali

# Preliminari

In [1]:
import pandas as pd
import os
from pathlib import Path

# Load Data

In [61]:
# Get data

data_path = Path('../data/base.tumoreprimitivo.csv/')
# Se il file non è presente, inserirlo manualmente prendendolo dalla cartella dropbox

data = pd.read_csv(data_path)

In [63]:
display(data.head())
print(f'{data.shape = }')

,id,profile,discrepanze_rilevate,motivazioni_discrepanze,radiologist,patient_id,sesso,data_nascita,interpretazioni,report_text,...,coinvolgimento_riflessione_peritoneale,coinvolgimento_fascia_mesorettale,distanza_minima_fascia_ore,linfonodi_sospetti,sedi_locoregionali,sedi_non_locoregionali,depositi_tumorali,numero_depositi,emvi_esteso,status
0,220,PietroPaoloAzzaro,"['n_classification', 'depositi_tumorali']",nessuna,NaN,35426899,M,1975-09-20,NaN,RM ADDOME INFERIORE (S/C MDC)\r\n\r\n\r\nL'ESA...,...,no,si,NaN,4.0,"['mesorettali', 'sacrali', 'mesenterici_inferi...",[],si,0.0,si,complete
1,219,PietroPaoloAzzaro,['nessuna'],nessuna,NaN,35430586,F,1946-08-05,NaN,RM ADDOME INFERIORE\r\n \r\nL'ESAME E STATO ES...,...,si,si,NaN,4.0,"['mesorettali', 'mesenterici_inferiori']",[],NaN,0.0,no,complete
2,218,PietroPaoloAzzaro,['nessuna'],nessuna,NaN,35461383,M,1967-01-28,NaN,RM ADDOME INFERIORE (S/C MDC)\r\n \r\nL'ESAME ...,...,no,no,NaN,6.0,"['mesorettali', 'iliaci_interni']",['inguinali'],NaN,0.0,no,complete
3,217,PietroPaoloAzzaro,['nessuna'],nessuna,NaN,35533934,F,1958-05-28,NaN,RM ADDOME INFERIORE (S/C MDC)\r\n\r\nL'ESAME E...,...,si,si,NaN,4.0,"['mesorettali', 'otturatori']",[],NaN,0.0,no,complete
4,216,PietroPaoloAzzaro,['nessuna'],nessuna,NaN,35543446,M,1939-06-15,NaN,RM ADDOME INFERIORE (S/C MDC)\r\n\r\nL'ESAME È...,...,no,si,NaN,4.0,"['mesorettali', 'mesenterici_inferiori']",[],no,0.0,no,complete


data.shape = (173, 43)


# Elimina righe doppie

In [66]:
help(pd.DataFrame.duplicated)

Help on function duplicated in module pandas.core.frame:

duplicated(self, subset: 'Hashable | Sequence[Hashable] | None' = None, keep: 'DropKeep' = 'first') -> 'Series'
    Return boolean Series denoting duplicate rows.
    
    Considering certain columns is optional.
    
    Parameters
    ----------
    subset : column label or sequence of labels, optional
        Only consider certain columns for identifying duplicates, by
        default use all of the columns.
    keep : {'first', 'last', False}, default 'first'
        Determines which duplicates (if any) to mark.
    
        - ``first`` : Mark duplicates as ``True`` except for the first occurrence.
        - ``last`` : Mark duplicates as ``True`` except for the last occurrence.
        - False : Mark all duplicates as ``True``.
    
    Returns
    -------
    Series
        Boolean series for each duplicated rows.
    
    See Also
    --------
    Index.duplicated : Equivalent method on index.
    Series.duplicated : Equiv

In [67]:
# Intere righe duplicate. Keep last perchè i report hanno id decrescente
duplicati = data.iloc[:, 1:].duplicated(keep='last')
print(f'Numero righe duplicate: {duplicati.sum()}')

data_clean = data[duplicati == False]
print('Righe doppie eliminate')

# Rimuovi righe con report duplicati
duplicati = data_clean['report_text'].duplicated(keep='last')
print(f'Numero righe con stesso referto: {duplicati.sum()}')

data_clean = data_clean[duplicati == False]
print('Righe eliminate')

data_clean.reset_index(inplace=True, drop=True)

print(f'{data_clean.shape = }')

Numero righe duplicate: 15
Righe doppie eliminate
Numero righe con stesso referto: 10
Righe eliminate
data_clean.shape = (148, 43)


# Analisi preliminari delle colonne

In [68]:
print(data_clean.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 43 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   id                                      148 non-null    int64  
 1   profile                                 148 non-null    object 
 2   discrepanze_rilevate                    148 non-null    object 
 3   motivazioni_discrepanze                 148 non-null    object 
 4   radiologist                             2 non-null      object 
 5   patient_id                              148 non-null    int64  
 6   sesso                                   146 non-null    object 
 7   data_nascita                            147 non-null    object 
 8   interpretazioni                         40 non-null     object 
 9   report_text                             148 non-null    object 
 10  morfologia                              141 non-null    object

## Reports

In [69]:
# Stampa alcuni referti
if False:
    for r in data_clean['report_text'].head(2):
        print(r)
        print(f"\n{100*'-'}\n")    

## Colonne extra
Colonne non rilevanti per l'allenamento del modello

In [94]:
for col in ['profile', 'discrepanze_rilevate', 'motivazioni_discrepanze', 'radiologist', 'sesso', 'motivazioni_extra', 'status']:
    print(data_clean[col].value_counts())
    print("\n")

profile
GuidoImbemba         75
PietroPaoloAzzaro    72
IlariaNacci           1
Name: count, dtype: int64


discrepanze_rilevate
['nessuna']                                  138
['n_classification', 'depositi_tumorali']     10
Name: count, dtype: int64


motivazioni_discrepanze
nessuna                                                    147
sono presenti depositi tumorali ma 5 linfonodi sospetti      1
Name: count, dtype: int64


radiologist
Arcioni Daniel    1
Barbaro           1
Name: count, dtype: int64


sesso
M    85
F    61
Name: count, dtype: int64


motivazioni_extra
nessuna    148
Name: count, dtype: int64


status
incomplete    75
complete      73
Name: count, dtype: int64




Eventualmente escludere righe con discrepanze

In [97]:
if False:
    data_clean[data_clean.discrepanze_rilevate == "['n_classification', 'depositi_tumorali']"].T

## Colonne target non numeriche

In [100]:
for col in ['morfologia', 'posizione', 'carcinosi_peritoneale', 'lesioni_ossee', 'riflessione_peritoneale_anteriore',
            'infiltrazione_tessuto_adiposo', 'infiltrazione_sfinteri',
            'infiltrazione_organi_extra', 'infiltrazione_organi_dettagli',
            'coinvolgimento_riflessione_peritoneale', 'coinvolgimento_fascia_mesorettale',
            'sedi_locoregionali', 'sedi_non_locoregionali', 'depositi_tumorali', 'emvi_esteso']:
    print(data_clean.fillna('NaN')[col].value_counts())
    print("\n")

morfologia
solido_anulare      57
solido_polipoide    56
mucinoso            28
NaN                  7
Name: count, dtype: int64


posizione
medio        67
basso        41
alto         24
NaN           9
giunzione     7
Name: count, dtype: int64


carcinosi_peritoneale
no     91
NaN    57
Name: count, dtype: int64


lesioni_ossee
no     88
NaN    58
si      2
Name: count, dtype: int64


riflessione_peritoneale_anteriore
cavallo    58
NaN        47
sotto      43
Name: count, dtype: int64


infiltrazione_tessuto_adiposo
si_5mm_plus    98
si_5mm         35
no              7
sospetto        4
NaN             4
Name: count, dtype: int64


infiltrazione_sfinteri
no                       132
NaN                       12
interno                    2
interno_piano_esterno      2
Name: count, dtype: int64


infiltrazione_organi_extra
no          104
si           19
sospetto     14
NaN          11
Name: count, dtype: int64


infiltrazione_organi_dettagli
NaN                                      

Commenti:
 - carcinosi peritoneale ha solo valori no (91) e None (57)
 - lesioni ossee è fortemente sbilanciata si 2, no 88, None 58
 - riflessione peritoneale anteriore non presenta il valore "sopra"
 - infiltrazione sfinteri è fortemente sbilanciata
 - infiltrazione_organi_dettagli è troppo poco utilizzata per trarne conoscenza
 - 


## Colonne target numeriche

In [103]:
colonne_target_numeriche = ['ore_inizio', 'ore_fine', 'dimensione_dll',
    'dimensione_dap', 'spessore_parietale', 'estensione_cranio_caudale',
    'distanza_oai', 'distanza_minima_fascia_ore',
    'linfonodi_sospetti', 'sedi_locoregionali', 'sedi_non_locoregionali',
    'depositi_tumorali', 'numero_depositi']

In [104]:
data_clean[colonne_target_numeriche].describe().T

,count,mean,std,min,25%,50%,75%,max
ore_inizio,23.0,6.608696,4.075862,1.0,3.00,6.0,10.50,12.0
ore_fine,24.0,7.708333,3.939534,1.0,4.00,8.5,12.00,12.0
dimensione_dll,8.0,36.250000,19.248377,15.0,21.50,34.0,42.25,72.0
dimensione_dap,7.0,41.857143,24.869182,21.0,30.00,31.0,43.00,95.0
spessore_parietale,30.0,20.100000,13.131562,7.0,12.00,15.0,22.00,60.0
estensione_cranio_caudale,140.0,49.821429,18.268291,18.0,36.75,49.0,60.00,130.0
distanza_oai,115.0,48.626087,24.722481,0.0,30.00,50.0,67.00,110.0
distanza_minima_fascia_ore,15.0,6.933333,3.575046,1.0,4.00,7.0,9.00,12.0
linfonodi_sospetti,147.0,2.795918,2.418362,0.0,0.00,3.0,4.00,12.0
numero_depositi,146.0,0.027397,0.351258,-1.0,0.00,0.0,0.00,3.0


escludere le segenti colonne perchè hanno troppi valori nulli:
- ore_inizio
- ore_fine
- dimensione_dll
- dimensione_dap
- distanza_minima_fascia_ore

## Colonne conclusioni

In [106]:
for col in ['stadio_T', 'stadio_N', 'stadio_N1c', 'mrf', 'emvi', 'metastasi']:
    print(data_clean.fillna('Nan')[col].value_counts())
    print("\n")

stadio_T
T3cd    49
T4b     34
T3ab    31
T4a     20
Nan      9
T1-2     5
Name: count, dtype: int64


stadio_N
N2a    62
N1b    26
N0     24
N+     21
N2b     6
N1a     6
Nan     2
N1c     1
Name: count, dtype: int64


stadio_N1c
False    125
True      23
Name: count, dtype: int64


mrf
-      87
+      59
Nan     2
Name: count, dtype: int64


emvi
-      103
+       35
Nan     10
Name: count, dtype: int64


metastasi
MX     77
Nan    48
M1     23
Name: count, dtype: int64


